# Stage 1 — 도시 장애물 환경 학습 (로컬 GPU)

**담당**: 이재왕 (work/evader)  
**씬**: `Assets/01. Scenes/Stage1.unity`  
**목표**: 도시 환경에서 장애물 회피 + GoalZone 도달 학습 (Pursuer 없음)  
**전략**: Stage1-A (`_goalOnlyMode=true`, `_currentStage=1`)  
**수렴 기준**: `goal_reach_rate ≥ 30%`, `crash_rate ≤ 15%` → Stage1-B (RL Pursuer 추가)  

---

## 사전 준비 (최초 1회)

```bash
cd c:\IIT_DroneLearning
.venv\Scriptsctivate
jupyter notebook python/notebooks/stage1_obstacle_local.ipynb
```

---

---
## 1. 환경 확인

In [1]:
import sys
import torch
import mlagents_envs

print(f'Python   : {sys.version.split()[0]}')
print(f'torch    : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'mlagents : {mlagents_envs.__version__}')


Python   : 3.10.11
torch    : 2.7.0+cu128
CUDA     : True
GPU      : NVIDIA GeForce RTX 5060
VRAM     : 8.5 GB
mlagents : 1.2.0.dev0


---
## 2. 경로 및 실험 설정

In [1]:
import os
from pathlib import Path

REPO_PATH  = Path("c:/IIT_DroneLearning")
LOG_DIR    = REPO_PATH / "python" / "results"
CONFIG_DIR = REPO_PATH / "python" / "config"

LOG_DIR.mkdir(parents=True, exist_ok=True)

SEED      = 42
INIT_FROM = "evader_s0_flat_44d_v6_seed42"  # Stage0 v6 warm-start (no goal-avoidance bias)
RUN_ID    = f"evader_s1_obstacle_44d_v4_seed{SEED}"

# True  -> --resume / False -> --force (fresh start with INIT_FROM)
RESUME = False

print(f"Repo      : {REPO_PATH}")
print(f"Log dir   : {LOG_DIR}")
print(f"Run ID    : {RUN_ID}")
if RESUME:
    print(f"Mode      : RESUME")
else:
    print(f"Mode      : FORCE  (warm-start from: {INIT_FROM})")


---
## 3. Config 확인

In [2]:
config_path = CONFIG_DIR / 'evader_s1_obstacle_template.yaml'
assert config_path.exists(), f'Config 없음: {config_path}'

print(f'Config: {config_path}{"=" * 60}')
print(config_path.read_text(encoding='utf-8'))


Config: c:\IIT_DroneLearning\python\config\evader_s1_obstacle_template.yaml============================================================
# evader_s1_obstacle_template.yaml
# Stage1-A v3: Curriculum Phase1 - near-range (SpawnCenter radius=12m)
# Goal: Mean Reward > +1.0 within 100k steps, then expand range
#
# CURRICULUM APPROACH:
#   Phase1 (v3): SpawnCenter radius=12m  -> max spawn-goal dist ~24m
#   Phase2 (v4): SpawnCenter radius=25m  -> max ~50m (after Phase1 converges)
#   Phase3 (v5): SpawnCenter radius=50m  -> full city range
#
# Unity Inspector changes from v2:
#   EpisodeSpawnCoordinator:
#     Strategy        = SpawnCenterRandom  (was: CityMetadata)
#     _minSeparation  = 3                  (was: 10)
#   SpawnCenter:
#     Auto Sync From City = OFF            (prevent metadata override)
#     Sync Mode       = Synchronized
#     Shared Range -> Radius = 12          (was: ~50)
#   EvaderAgent:
#     _goalOnlyMode          = true
#     _currentStage          = 1
#     Max Episo

---
## 4. Unity Editor 연결 확인

> ⚠️ 다음 셀 실행 **전** Unity Editor에서 `Stage1.unity` 씬을 열어두세요.  
> 셀 실행 → `mlagents-learn`이 포트 5004 대기 → Unity에서 ▶ Play 누르면 자동 연결.

**Unity Inspector 체크리스트** (학습 전 반드시 확인):
- `Drone_Evader` → EvaderAgent: `_currentStage = 1`, `_goalOnlyMode = true`, `Max Episode Seconds = 35`
- `Drone_Evader` → EvaderReward:
  - `_goalShapingCoeff = 0.3`
  - `_goalProximityRange = 20.0` (20m 이내 근접 보너스)
  - `_goalProximityCoeff = 0.005`
  - `_velAlignCoeff = 0.003` (호버링 방지 최소 압력)
  - `_yawAlignCoeff = 0.0`
  - `_proximityThreshold = 0.15`, `_proximityCoeff = 0.002`
  - `_heightWarningY = 17`, `_heightPenaltyCoeff = -0.02`
  - `_timePenaltyPerStep = -0.001`
- `Drone_Evader` → BehaviorParameters: `Behavior Name = Drone_Evader`, `Behavior Type = Default`
- DroneDepthSystem: **Disabled**, DroneVisionSystem: **Enabled** (RGB 3ch)


In [3]:
import socket

port = 5004
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    result = s.connect_ex(('127.0.0.1', port))
    if result == 0:
        print(f'⚠️  포트 {port} 이미 사용 중 — 이전 mlagents-learn 프로세스가 남아있을 수 있습니다.')
    else:
        print(f'✅ 포트 {port} 사용 가능. 다음 셀 실행 후 Unity에서 Play 하세요.')


✅ 포트 5004 사용 가능. 다음 셀 실행 후 Unity에서 Play 하세요.


---
## 5. 학습 실행

> 이 셀을 실행하면 `mlagents-learn`이 Unity 연결을 대기합니다.  
> **Unity Editor → Stage1.unity → ▶ Play** 를 누르면 학습이 시작됩니다.


In [ ]:
import subprocess, sys, socket, time, threading
from pathlib import Path

venv_scripts = Path(sys.executable).parent
mlagents_bin = venv_scripts / 'mlagents-learn.exe'

# RESUME=True: --resume (이어받기) / False: --force + warm-start
cmd = [
    str(mlagents_bin), str(config_path),
    f'--run-id={RUN_ID}',
    f'--results-dir={LOG_DIR}',
]
if RESUME:
    cmd += ['--resume']
    print('Mode: RESUME — 마지막 체크포인트에서 이어받습니다.')
else:
    cmd += ['--force']
    if INIT_FROM:
        cmd += [f'--initialize-from={INIT_FROM}']
        print(f'Mode: FORCE — Warm-start from: {INIT_FROM}')
    else:
        print('Mode: FORCE — 처음부터 학습합니다.')

print('실행 커맨드:')
print(' '.join(str(c) for c in cmd))
print()
print('⏳ mlagents-learn 시작 중... (포트 5004 준비 대기)')

output_lines = []

def stream_output(pipe):
    for line in pipe:
        text = line.decode('utf-8', errors='replace').rstrip()
        output_lines.append(text)
        print(text)

proc = subprocess.Popen(
    cmd, cwd=str(REPO_PATH),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
t = threading.Thread(target=stream_output, args=(proc.stdout,), daemon=True)
t.start()

port = 5004
ready = False
for _ in range(60):
    time.sleep(0.5)
    if proc.poll() is not None:
        t.join(timeout=2)
        print(f'\n❌ mlagents-learn이 즉시 종료되었습니다 (exit={proc.returncode})')
        break
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.3)
        if s.connect_ex(('127.0.0.1', port)) == 0:
            ready = True
            break

if ready:
    print()
    print('=' * 55)
    print('✅ mlagents-learn 준비 완료 — 포트 5004 listening')
    print('▶  지금 Unity Editor에서 Stage1.unity → Play 누르세요!')
    print('=' * 55)
    proc.wait()
    t.join()
    print(f'\n학습 완료 (exit={proc.returncode})')
else:
    proc.terminate()
    t.join(timeout=2)
    raise RuntimeError('mlagents-learn 시작 실패 — 위 출력 내용을 확인하세요.')


Mode: RESUME — 마지막 체크포인트에서 이어받습니다.
실행 커맨드:
C:\IIT_DroneLearning\.venv\Scripts\mlagents-learn.exe c:\IIT_DroneLearning\python\config\evader_s1_obstacle_template.yaml --run-id=evader_s1_obstacle_44d_v3_seed42 --results-dir=c:\IIT_DroneLearning\python\results --resume

⏳ mlagents-learn 시작 중... (포트 5004 준비 대기)
[INFO] Listening on port 5004. Start training by pressing the Play button in the Unity Editor.

✅ mlagents-learn 준비 완료 — 포트 5004 listening
▶  지금 Unity Editor에서 Stage1.unity → Play 누르세요!
[INFO] Connected to Unity environment with package version 4.0.0 and communication version 1.5.0
[INFO] Connected new brain: Drone_Evader?team=0


	Unity Technologies

 Version information:
  ml-agents: 1.2.0.dev0,
  ml-agents-envs: 1.2.0.dev0,
  Communicator API: 1.5.0,
  PyTorch: 2.7.0+cu128
[INFO] Hyperparameters for behavior name Drone_Evader:
	trainer_type:	ppo
	hyperparameters:
	  batch_size:	128
	  buffer_size:	20480
	  learning_rate:	0.0002
	  beta:	0.003
	  epsilon:	0.2
	  lambd:	0.99
	  num_

---
## 6. TensorBoard 모니터링

학습 중 **새 터미널**에서 아래 명령어로 TensorBoard를 실행하세요.


In [ ]:
tb_cmd = f'.venv\Scripts\tensorboard --logdir python/results/{evader_s1_obstacle_44d_v1_seed42} --port 6006'
print('새 터미널에서 실행:')
print(tb_cmd)
print()
print('브라우저: http://localhost:6006')
print()
print('주요 모니터링 지표:')
print('  Environment/Cumulative Reward  → 상승 추세 확인 (초반 1~3 기대)')
print('  Environment/Episode Length     → 500~1000 (탐색 중 정상)')
print('  Policy/Entropy                 → 서서히 감소해야 함')
print()
print('⚠️  위험 신호:')
print('  - 50k 스텝 이후에도 Mean Reward 음수 지속 → 보상 재설계 필요')
print('  - Episode Length 항상 Max(1250) → 타임아웃 과다 (crash 또는 방황)')


---
## 7. 결과 확인 및 ONNX 경로

In [ ]:
run_dir = LOG_DIR / RUN_ID

if run_dir.exists():
    onnx_files = list(run_dir.glob('**/*.onnx'))
    pt_files   = list(run_dir.glob('**/*.pt'))

    print(f'✅ 결과 폴더: {run_dir}')
    print(f'ONNX 파일 ({len(onnx_files)}개):')
    for f in sorted(onnx_files):
        print(f'  {f.name}')
    print(f'체크포인트 ({len(pt_files)}개): {len(pt_files)}개')
else:
    print(f'❌ 결과 폴더 없음: {run_dir}')
    print('5번 셀(학습)을 먼저 실행하세요.')


---
## 8. Stage1-B 전환 (수렴 확인 후)

수렴 기준:
- `goal_reach_rate ≥ 30%`
- `crash_rate ≤ 15%`

**Unity Inspector 변경 (Stage1-B):**
- `EvaderAgent._goalOnlyMode` = **false** (RL Pursuer 활성)
- Pursuer 오브젝트 → BehaviorParameters → Model = `pursuer_s2_catch_v3_499980.onnx`

**다음 버전 설정:**
```python
INIT_FROM = 'evader_s1_obstacle_44d_v1_seed42'
RUN_ID    = f'evader_s1_pursuer_44d_v1_seed{SEED}'
```
